## Leveraging Word Positions

While **token embeddings** are suitable for representing token identities, they do **not contain any information about token order**. This presents a problem:

> ⚠️ Limitation of Self-Attention
>
> The transformer’s self-attention mechanism is **invariant to sequence order**. It treats inputs as an unordered set.
> This is powerful for modeling pairwise relationships, but it **cannot distinguish** between:
>
> * `"John saw Mary"` and
> * `"Mary saw John"`
>
> unless **additional information about token positions** is provided.

Thus, transformers require a **second embedding layer** that encodes positional information. The resulting input to the transformer is:

$$
\text{Final Input} = \text{Token Embedding} + \text{Positional Embedding}
$$

---

## Types of Positional Embeddings

We categorize positional encodings into two main paradigms:

### Absolute Positional Embeddings

> Each position $p \in \{0, 1, 2, ..., T-1\}$ is assigned a fixed or learned vector $\mathbf{p}_p \in \mathbb{R}^d$, and added to the token embedding at that position.

* Used in the **original Transformer (Vaswani et al., 2017)** via a **fixed sinusoidal formula**.
* Used in **OpenAI GPT models**, but with **learnable** positional embeddings instead.
* Sensitive to **exact token locations**.

#### Pros

* Precise localization.
* Easy to implement and interpret.

#### Cons

* Limited extrapolation to longer sequences.
* Fixed maximum context length.

---

### Relative Positional Embeddings

> Instead of encoding the absolute index of each token, this approach encodes the **distance between token pairs** (e.g., token $i$ is 3 positions ahead of token $j$).

* Popularized by **Transformer-XL** and **DeBERTa**.
* Captures **pairwise token distance** rather than position.
* Often implemented as **bias terms in attention scores**.

#### Pros

* Better generalization to unseen sequence lengths.
* Enables flexible modeling of relative linguistic structures (e.g., subject-verb-object).

#### Cons

* More complex to implement.
* May require more computation in attention layers.

---

### Other Positional Embedding Strategies

| Strategy                                 | Description                                                     | Used By                |
| ---------------------------------------- | --------------------------------------------------------------- | ---------------------- |
| **Rotary Position Embedding (RoPE)**     | Injects position into attention **via rotation of Q/K vectors** | LLaMA, GPT-NeoX, PaLM  |
| **ALiBi (Attention with Linear Biases)** | Applies position-based bias directly in attention scores        | T5.1.1, Mistral, GPT-J |
| **Hyena Position Encoding**              | Structured position encoding in state-space models              | Hyena (DeepMind)       |
| **Neural Positional Embeddings**         | Learned through neural nets (e.g., MLPs) rather than embeddings | Some experimental work |
| **No Positional Encoding**               | Omitted entirely in recurrence-based models                     | RWKV                   |

---

## Examples of Positional Embedding Usage in Major Models

| Model Family   | Positional Encoding Type | Learnable? | Comment                                   |
| -------------- | ------------------------ | ---------- | ----------------------------------------- |
| GPT-2, GPT-3   | Absolute                 | Yes        | Learned during training                   |
| GPT-4, GPT-4.1 | Absolute + RoPE (hybrid) | Yes        | Likely learned + efficient generalization |
| BERT           | Absolute                 | Yes        | Learned positional vectors                |
| Transformer-XL | Relative                 | Yes        | Enables better memory integration         |
| T5             | Relative + ALiBi         | Yes        | Efficient during decoding                 |
| LLaMA          | RoPE                     | No         | Rotational embedding (fixed function)     |
| DeBERTa        | Relative                 | Yes        | Relative distance with disentangled heads |
| Mistral        | ALiBi                    | No         | Lightweight and fast decoding             |

---

## Create Initial Positional Embeddings (Absolute, Learnable)

We'll define a PyTorch `nn.Embedding` layer to store positional embeddings—just like token embeddings—but with one embedding per possible position in the sequence.

In [1]:
import torch
import torch.nn as nn

# Set context window size and embedding dimension
max_seq_len = 16         # supports sequences up to 16 tokens
embedding_dim = 64       # must match token embedding dimension

# Create learnable absolute positional embeddings
positional_embedding = nn.Embedding(num_embeddings=max_seq_len, embedding_dim=embedding_dim)

# Inspect the matrix
print("Positional Embedding Matrix:")
print(positional_embedding.weight)

Positional Embedding Matrix:
Parameter containing:
tensor([[-1.1896,  2.7186,  0.6888,  ..., -0.4475, -0.4634,  0.1896],
        [ 1.3297,  1.2234, -0.3012,  ..., -2.1098, -0.0630,  1.5648],
        [ 1.1708, -1.2097,  0.0422,  ...,  1.7480, -0.5136,  0.5146],
        ...,
        [ 0.3715,  0.4250,  0.3263,  ..., -0.9780, -1.9382,  0.5251],
        [-0.9100,  0.2797, -0.9824,  ..., -0.7483, -2.0664,  1.3096],
        [-1.0057,  1.5535,  0.2352,  ...,  0.5751,  0.2587,  0.4178]],
       requires_grad=True)


> This creates a matrix of shape $(16, 64)$, where:
>
> * Each **row** corresponds to a position $p \in \{0, \dots, 15\}$
> * Each **column** is a feature of the position embedding
> * The values are initialized randomly and optimized via backpropagation during training.

---

### Combining Token and Positional Embeddings

To prepare the full input for a transformer block:

In [4]:
import torch
import torch.nn as nn

# Simulate token IDs for one sequence
token_ids = torch.tensor([3, 7, 5, 1, 0])             # sequence length = 5

# Define hyperparameters
vocab_size = 1000       # set according to your tokenizer
embedding_dim = 64      # dimensionality of embedding vectors
max_seq_len = 512       # maximum length your model can handle

# Define embedding layers
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
positional_embedding = nn.Embedding(num_embeddings=max_seq_len, embedding_dim=embedding_dim)

# Create position IDs automatically
position_ids = torch.arange(len(token_ids))

# Get embeddings
token_vectors = embedding_layer(token_ids)            # (seq_len, embedding_dim)
position_vectors = positional_embedding(position_ids) # (seq_len, embedding_dim)

# Add element-wise
final_input = token_vectors + position_vectors

# print("Final Input to Transformer:")
# print(final_input)

This is the input to the first transformer block. The sum is computed element-wise and **broadcasted across batches** if needed.

Code Walk-Through

## **Part 1: Understanding the Shapes in Your Code**

```python
token_ids = torch.tensor([3, 7, 5, 1, 0])  # shape: (5,)
```

* This is a 1D tensor containing **5 token IDs** (a single sequence of length 5).
* Shape: `(seq_len,)` = `(5,)`

```python
embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim)
```

* This maps each token ID to a dense vector of size `embedding_dim = 64`.
* The embedding matrix inside has shape: `(vocab_size, embedding_dim)` = `(1000, 64)`

```python
token_vectors = embedding_layer(token_ids)
```

* You're feeding a 1D tensor of 5 token IDs into the embedding layer.
* Output shape: `(seq_len, embedding_dim)` = `(5, 64)`

```python
position_ids = torch.arange(len(token_ids))  # [0, 1, 2, 3, 4]
position_vectors = positional_embedding(position_ids)
```

* `position_ids`: shape `(5,)`
* `positional_embedding`: lookup table of shape `(max_seq_len, embedding_dim)` = `(512, 64)`
* `position_vectors`: shape `(5, 64)`

```python
final_input = token_vectors + position_vectors
```

* Both operands have shape `(5, 64)`; element-wise addition is valid.
* `final_input`: shape `(5, 64)`

---

## **Part 2: Scaling Up with `DataLoader`**

Now consider you have a full dataset and use a `DataLoader` like this:

```python
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
```

If your `DataLoader` is yielding `(x, y)` pairs for next-token prediction:

* `inputs` is a tensor of shape `(batch_size, window_size)`
* `targets` is a tensor of shape `(batch_size, window_size)`

### Example:

```python
batch_size = 32
window_size = 128
```

Then:

* `inputs.shape` = `(32, 128)`
* `targets.shape` = `(32, 128)`

### Embedding:

```python
token_embeddings = embedding_layer(inputs)
```

* Input shape: `(batch_size, window_size)`
* Output shape: `(batch_size, window_size, embedding_dim)` = `(32, 128, 64)`

### Positional Embedding:

```python
position_ids = torch.arange(window_size).unsqueeze(0)  # shape: (1, 128)
position_vectors = positional_embedding(position_ids)  # shape: (1, 128, 64)
```

* You broadcast position vectors across the batch.

### Final Input:

```python
final_input = token_embeddings + position_vectors  # shape: (32, 128, 64)
```

---

## **Summary Table of Shapes**

| Object                   | Description                 | Shape                                      |
| ------------------------ | --------------------------- | ------------------------------------------ |
| `token_ids`              | Sequence of token IDs       | `(seq_len,)`                               |
| `token_vectors`          | Token embeddings            | `(seq_len, embedding_dim)`                 |
| `position_ids`           | Position indices            | `(seq_len,)`                               |
| `position_vectors`       | Positional embeddings       | `(seq_len, embedding_dim)`                 |
| `final_input`            | Input to transformer        | `(seq_len, embedding_dim)`                 |
| `inputs` (from loader)   | Batch of input sequences    | `(batch_size, window_size)`                |
| `token_embeddings`       | Embedded batch input        | `(batch_size, window_size, embedding_dim)` |
| `position_ids` (batched) | Same positions across batch | `(1, window_size)`                         |
| `position_vectors`       | Pos embeddings broadcasted  | `(1, window_size, embedding_dim)`          |
| `final_input` (batched)  | Final model input           | `(batch_size, window_size, embedding_dim)` |

---

## Final Tip: Why This Matters

* Transformers **do not inherently understand order**. That is why positional embeddings must be added.
* During training, you always add token and positional embeddings **before feeding into the Transformer**.
* Understanding shapes helps you debug, especially when batching and ensuring correct broadcast behavior.

Let me know if you want visual diagrams or PyTorch model classes that integrate these components.




## 1. Implementing **Fixed Sinusoidal Positional Encodings** (from the original Transformer)

### Background

In the original "Attention is All You Need" paper (Vaswani et al., 2017), the authors proposed **fixed sinusoidal positional encodings** to allow the model to use **position-specific but non-learned features**.

The idea is to encode positions using sinusoids of varying frequencies:

$$
\text{PE}_{p, 2i} = \sin\left(\frac{p}{10000^{2i / d}}\right), \quad
\text{PE}_{p, 2i+1} = \cos\left(\frac{p}{10000^{2i / d}}\right)
$$

Where:

* $p$: position index
* $i$: embedding dimension index
* $d$: total embedding dimension

These encodings can **generalize to longer sequences** without learning new parameters.

### PyTorch Implementation

```python
import torch
import math

def get_sinusoidal_positional_encoding(max_len: int, embedding_dim: int) -> torch.Tensor:
    """
    Generate a fixed sinusoidal positional encoding matrix.
    Shape: (max_len, embedding_dim)
    """
    pos = torch.arange(max_len).unsqueeze(1)                   # (max_len, 1)
    i = torch.arange(embedding_dim).unsqueeze(0)               # (1, embedding_dim)

    angle_rates = 1 / torch.pow(10000, (2 * (i // 2)) / embedding_dim)
    angle_rads = pos * angle_rates                             # broadcasted (max_len, embedding_dim)

    pe = torch.zeros_like(angle_rads)
    pe[:, 0::2] = torch.sin(angle_rads[:, 0::2])               # even indices
    pe[:, 1::2] = torch.cos(angle_rads[:, 1::2])               # odd indices

    return pe
```

### Example Usage

```python
max_len = 16
d_model = 64
sin_pe = get_sinusoidal_positional_encoding(max_len, d_model)

print("Fixed Sinusoidal Encoding (16 × 64):")
print(sin_pe)
```

---

## 2. How **RoPE** (Rotary Position Embedding) Works Mathematically and Geometrically

### Conceptual Overview

**RoPE** is a strategy where positional information is introduced **not by addition**, but by **rotating the Q and K vectors** in the attention mechanism according to their positions.

RoPE replaces:

$$
\text{Q}_i = \text{Q}_i + \text{PE}_i, \quad \text{K}_i = \text{K}_i + \text{PE}_i
$$

with:

$$
\text{Q}_i^\text{rotated} = R_i(\text{Q}_i), \quad \text{K}_i^\text{rotated} = R_i(\text{K}_i)
$$

Where $R_i(\cdot)$ is a **position-dependent rotation operator** applied in complex space or interleaved real pairs.

### Mathematical Details

RoPE operates on **even and odd dimensions as coordinate pairs** and applies a rotation matrix:

$$
\begin{bmatrix}
x_{2k}^\prime \\
x_{2k+1}^\prime
\end{bmatrix}
=
\begin{bmatrix}
\cos(\theta_k p) & -\sin(\theta_k p) \\
\sin(\theta_k p) & \cos(\theta_k p)
\end{bmatrix}
\cdot
\begin{bmatrix}
x_{2k} \\
x_{2k+1}
\end{bmatrix}
$$

Where $\theta_k = 10000^{-2k/d}$, and $p$ is the position.

### Intuition

* This is equivalent to rotating each coordinate-pair (2D subspace) of the query/key vectors.
* The dot product $\text{Q}_i^\top \text{K}_j$ becomes **position-aware**, since rotation angles depend on $i$ and $j$.
* RoPE enables relative information to be encoded **implicitly** via inner products.

---

## 3. Benchmarking Positional Strategies: Accuracy and Generalization

Let’s summarize empirical observations from research on the effects of positional encoding methods on downstream performance:

| Encoding Method        | Learnable | Relative or Absolute | Extrapolates to Long Seqs | Commonly Used In     | Comments                     |
| ---------------------- | --------- | -------------------- | ------------------------- | -------------------- | ---------------------------- |
| **Fixed Sinusoidal**   | ❌         | Absolute             | ❌ (limited)               | Original Transformer | Lightweight, but rigid       |
| **Learnable Absolute** | ✅         | Absolute             | ❌                         | GPT, BERT            | Good for fixed-length        |
| **Relative Biases**    | ✅         | Relative             | ✅                         | Transformer-XL, T5   | Handles variable lengths     |
| **ALiBi**              | ❌         | Relative (additive)  | ✅✅                        | Mistral, GPT-J       | Very efficient and effective |
| **RoPE**               | ❌         | Relative             | ✅✅✅                       | LLaMA, GPT-NeoX      | Excellent for long contexts  |
| **Neural Position**    | ✅         | Absolute             | ❓                         | Experimental         | Requires more compute        |

> ✅ = performs well
> ✅✅ = state-of-the-art in practice
> ❌ = limitation
> ❓ = not well studied

---

## Next Steps

You now have:

1. **Implemented sinusoidal encodings**.
2. **Understood RoPE at a geometric level**.
3. **Seen how positional strategies compare empirically**.




# Advanced LLM Input Strategies

| Type                              | Use Case                                                                 | Typical Models                                   | Mechanism                                                                                      | Benefits                                                                                       | Common in LLMs?       |
|-----------------------------------|---------------------------------------------------------------------------|--------------------------------------------------|------------------------------------------------------------------------------------------------|------------------------------------------------------------------------------------------------|-----------------------|
| Token Embeddings                  | Convert token IDs into dense vectors representing semantic meaning.      | All transformer-based LLMs (e.g., GPT, BERT)     | Each token ID maps to a fixed vector via an embedding matrix.                                  | Core representation of input text; essential for language understanding.                      | Yes                   |
| Positional Embeddings             | Introduce order information into input sequences.                        | GPT, BERT, LLaMA, T5                             | Fixed (sinusoidal) or learned vectors added to token embeddings based on position.             | Allows attention mechanism to distinguish token positions.                                     | Yes                   |
| Segment / Sentence Type Embeddings | Differentiate between sentence pairs or text segments.                   | BERT, RoBERTa                                    | Adds a learned vector for each segment (e.g., A or B).                                          | Enables models to handle pairwise inputs (e.g., in QA, NLI).                                   | Yes (encoders)        |
| Task / Instruction Embeddings     | Condition model behavior on task instructions.                           | T5, FLAN, InstructGPT, ChatGPT                   | Special tokens or prompts prepended to inputs, sometimes with separate learned vectors.        | Supports multi-task learning and better generalization.                                        | Common                |
| Modality Embeddings               | Handle multiple input types like images, audio, or video.                | Flamingo, Kosmos, Gemini, LLaVA                  | Project modality-specific encoder outputs into shared space; add modality token.               | Enables multimodal reasoning and integration.                                                  | Increasing            |
| Time / Temporal Embeddings        | Capture sequence timing or periodicity for temporal tasks.               | Time-series Transformers, RecSys models          | Use timestamps or periodic functions to create temporal vectors.                               | Improves performance in time-sensitive tasks like forecasting.                                 | Contextual            |
| Document Layout Embeddings        | Use visual layout and structure in text processing.                      | LayoutLM, Longformer                             | Embed 2D coordinates or structural metadata into inputs.                                       | Boosts performance on forms, PDFs, tables, etc.                                                | Specialized           |
| Role / Speaker Embeddings         | Differentiate between users, assistants, or personas.                    | DialoGPT, ChatGPT, LLaMA Agents                  | Add role-specific embeddings or tags to turns in dialogue.                                     | Improves coherence and context tracking in dialogues.                                          | Dialogue-focused      |
| Retrieval-Augmented Inputs        | Incorporate external retrieved knowledge into model inputs.              | RETRO, RAG, LLaMAIndex                           | Embed or concatenate retrieved passages with special tokens.                                   | Increases factual accuracy and context length.                                                 | Increasing            |
| Prefix / Soft Prompt Embeddings   | Control or adapt model behavior without changing weights.                | P-Tuning v2, Prefix-Tuning, SoftPrompt-Tuning    | Insert learned virtual tokens into the input sequence.                                         | Lightweight and efficient fine-tuning method.                                                  | Efficient setups       |
| Memory / Cache Embeddings         | Support long-context modeling by reusing past information.               | Memorizing Transformers, Gemini 1.5, RWKV        | Summarized past context or recurrent states reused as input.                                   | Enables infinite or recurrent context processing.                                              | Emerging frontier     |
| Advanced Positional Strategies    | Improve positional encoding for long or flexible sequences.              | LLaMA (RoPE), LongNet (xPos), ALiBi, YaRN        | Rotary, linear bias, or extrapolatable encodings.                                              | Improves scaling to longer sequences; better extrapolation.                                    | State-of-the-art      |




## **Part 1: Visualizing Sinusoidal and RoPE Positional Encodings**

We will **plot and interpret** the geometry of:

1. **Fixed Sinusoidal Positional Encodings** — what the embedding matrix looks like.
2. **Rotary Position Embedding (RoPE)** — how rotation is applied to query/key vectors in attention.

### 1.1 Visualizing Sinusoidal Encoding Patterns

Let’s examine the first few rows of a sinusoidal positional encoding matrix.

```python
import torch
import matplotlib.pyplot as plt

def get_sinusoidal_encoding(max_len: int, d_model: int):
    pos = torch.arange(max_len).unsqueeze(1)
    i = torch.arange(d_model).unsqueeze(0)
    angle_rates = 1 / (10000 ** (2 * (i // 2) / d_model))
    angle_rads = pos * angle_rates
    pe = torch.zeros_like(angle_rads)
    pe[:, 0::2] = torch.sin(angle_rads[:, 0::2])
    pe[:, 1::2] = torch.cos(angle_rads[:, 1::2])
    return pe

# Create encoding matrix
max_len = 100
d_model = 16
pe = get_sinusoidal_encoding(max_len, d_model)

# Plot a few dimensions
plt.figure(figsize=(12, 6))
for dim in range(8):  # first 8 dimensions
    plt.plot(pe[:, dim], label=f"dim {dim}")
plt.title("Sinusoidal Positional Encoding (first 8 dimensions)")
plt.xlabel("Token position")
plt.ylabel("Encoding value")
plt.legend()
plt.grid(True)
plt.show()
```

> ### Interpretation
>
> * Even dimensions show sin curves; odd dimensions show cosines.
> * Lower dimensions oscillate slowly, higher ones faster.
> * Each position gets a **unique signature**, and differences are smooth and continuous.

---

### 1.2 Visualizing RoPE: 2D Rotation Behavior

Let’s simulate how a **position-dependent rotation** works on a 2D vector slice.

```python
import numpy as np

def rotate(vector, theta):
    """Apply 2D rotation to [x, y] vector."""
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    return rot_matrix @ vector

# Base vector (e.g., from query/key)
v = np.array([1.0, 0.0])
angles = np.linspace(0, np.pi, 8)

# Rotate and plot
plt.figure(figsize=(6, 6))
for angle in angles:
    v_rot = rotate(v, angle)
    plt.arrow(0, 0, v_rot[0], v_rot[1], head_width=0.05, alpha=0.7)
    plt.text(v_rot[0], v_rot[1], f"{round(angle, 2)} rad", fontsize=8)
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)
plt.title("RoPE: Position-Dependent Vector Rotations")
plt.grid(True)
plt.gca().set_aspect("equal")
plt.show()
```

> ### Interpretation
>
> * RoPE rotates pairs of vector components (e.g., \[x₀, x₁], \[x₂, x₃]) by an angle that **increases with position**.
> * These rotations cause the dot product between queries and keys to reflect **relative position**.
> * It’s **parameter-free**, differentiable, and consistent across varying sequence lengths.

---

## **Part 2: Implementing Relative Positional Embeddings**

Let’s now implement a basic **relative attention bias** mechanism inspired by Transformer-XL and T5. Instead of associating each token with a position, we associate **each query-key pair** with a **distance**.

### 2.1 Define Relative Bias Table

```python
class RelativePositionalBias(nn.Module):
    def __init__(self, max_distance: int, num_heads: int):
        super().__init__()
        self.max_distance = max_distance
        self.num_heads = num_heads
        self.bias_table = nn.Embedding(2 * max_distance + 1, num_heads)

    def forward(self, seq_len: int):
        # Compute relative distances: i - j for every pair
        positions = torch.arange(seq_len)
        rel_dist = positions[None, :] - positions[:, None]  # (seq_len, seq_len)
        rel_dist = rel_dist.clamp(-self.max_distance, self.max_distance)
        rel_dist += self.max_distance  # shift to [0, 2*max_dist]

        # Lookup bias values for each head
        bias = self.bias_table(rel_dist)  # shape: (seq_len, seq_len, num_heads)
        return bias.permute(2, 0, 1)      # shape: (num_heads, seq_len, seq_len)
```

### 2.2 Example Usage

```python
num_heads = 4
seq_len = 8
rel_bias = RelativePositionalBias(max_distance=4, num_heads=num_heads)
bias = rel_bias(seq_len)

print("Relative positional bias shape:", bias.shape)
# Output shape: (num_heads, seq_len, seq_len)
```

> ### How It Works
>
> * Each attention score is **adjusted by a bias** that depends on the **distance between tokens**.
> * Token pairs with relative distance 0 (e.g., self-attention) get one bias value; distance +1 another, and so on.
> * The model **learns** how to weight these distances during training.

---

## Summary

| Component               | Description                                               | PyTorch Module             |
| ----------------------- | --------------------------------------------------------- | -------------------------- |
| **Sinusoidal Encoding** | Fixed, interpretable, adds positional info by function    | Manual tensor              |
| **RoPE**                | Parameter-free, rotates queries/keys to reflect positions | Applied in attention heads |
| **Relative Bias**       | Learnable, adds bias based on query-key pair distance     | `RelativePositionalBias`   |

You now have the tools to **visualize**, **understand**, and **implement** every major category of positional encoding.

Would you like to see:

* How these encodings affect **attention score heatmaps**?
* A **side-by-side benchmark** on language modeling accuracy or extrapolation tasks?
* An **ablation study** to test removing positional encodings?

Let me know your preferred direction.
